# 03 · Columns & built-in functions

`pyspark.sql.functions` (imported as `F`) has **hundreds** of functions for maths, strings, dates, nulls,
conditionals… They run inside the JVM, so they're fast. **Rule #1 of PySpark: if a built-in function exists, use it.**

Full list: <https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html>

In [1]:
# Setup — run this cell first every time you open the notebook
import os, sys
os.environ.setdefault("JAVA_HOME", "/opt/homebrew/opt/openjdk@17")   # Spark runs on Java
os.environ["PYSPARK_PYTHON"] = sys.executable                         # workers use this same Python

from pyspark.sql import SparkSession, Window, functions as F, types as T

spark = (SparkSession.builder
         .appName("pyspark-course")
         .master("local[*]")                               # run locally, use all CPU cores
         .config("spark.sql.shuffle.partitions", "8")      # explained in lesson 10
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("FATAL")                   # keep Java's log noise out of the notebook

DATA, OUT = "../data", "../output"

In [2]:
customers = spark.read.csv(f"{DATA}/customers.csv", header=True, inferSchema=True)
orders = spark.read.csv(f"{DATA}/orders.csv", header=True, inferSchema=True)
orders.show(3)

+--------+-----------+----------+--------+-------------------+---------+
|order_id|customer_id|product_id|quantity|           order_ts|   status|
+--------+-----------+----------+--------+-------------------+---------+
|       1|         26|        27|       2|2024-09-11 08:18:52|completed|
|       2|          9|        23|       1|2024-04-18 15:22:01|completed|
|       3|         93|         4|       5|2024-04-01 16:43:45|completed|
+--------+-----------+----------+--------+-------------------+---------+
only showing top 3 rows


## Literals, arithmetic, aliases and casting

In [3]:
orders.select(
    "order_id",
    "quantity",
    (F.col("quantity") * 2).alias("double_qty"),
    F.lit("web").alias("channel"),                        # a constant value
    F.col("quantity").cast("string").alias("qty_str"),    # change type
    F.round(F.col("quantity") / 3, 2).alias("third"),
).show(3)

+--------+--------+----------+-------+-------+-----+
|order_id|quantity|double_qty|channel|qty_str|third|
+--------+--------+----------+-------+-------+-----+
|       1|       2|         4|    web|      2| 0.67|
|       2|       1|         2|    web|      1| 0.33|
|       3|       5|        10|    web|      5| 1.67|
+--------+--------+----------+-------+-------+-----+
only showing top 3 rows


### Spark 4 is strict about bad casts (ANSI mode)
Since Spark 4.0, *ANSI mode* is on by default: casting `"N/A"` to an int **raises an error** instead of
silently returning null. When the data might be dirty, use `try_cast` (and friends like `try_divide`,
`try_to_timestamp`) which return `null` on failure.

In [4]:
bad = spark.createDataFrame([("12",), ("N/A",), ("7",)], "raw string")
try:
    bad.select(F.col("raw").cast("int")).show()
except Exception as e:
    print("cast failed:", type(e).__name__)

bad.select("raw", F.col("raw").try_cast("int").alias("as_int")).show()

cast failed: NumberFormatException


+---+------+
|raw|as_int|
+---+------+
| 12|    12|
|N/A|  NULL|
|  7|     7|
+---+------+



## Conditional logic: `when` / `otherwise`
Spark's version of `if / elif / else` (or SQL's `CASE WHEN`). Conditions are checked in order; the first match wins.
Without `.otherwise(...)` non-matching rows get `null`.

In [5]:
orders.select(
    "order_id", "quantity", "status",
    F.when(F.col("status") == "completed", "✓")
     .when(F.col("status") == "returned", "↩")
     .otherwise("✗").alias("icon"),
).show(5)

+--------+--------+---------+----+
|order_id|quantity|   status|icon|
+--------+--------+---------+----+
|       1|       2|completed|   ✓|
|       2|       1|completed|   ✓|
|       3|       5|completed|   ✓|
|       4|       1|completed|   ✓|
|       5|       3|completed|   ✓|
+--------+--------+---------+----+
only showing top 5 rows


## String functions

In [6]:
customers.select(
    "name",
    F.upper("name").alias("upper"),
    F.length("name").alias("len"),
    F.split("name", " ").getItem(0).alias("first_name"),
    F.substring("name", 1, 3).alias("first3"),                     # 1-based!
    F.concat_ws(" | ", "name", "country").alias("joined"),
    F.regexp_replace("email", r"\d+", "#").alias("no_digits"),
    F.regexp_extract("email", r"^([a-z]+)\.", 1).alias("email_first"),
).show(5, truncate=False)

+------------+------------+---+----------+------+--------------------+------------------------+-----------+
|name        |upper       |len|first_name|first3|joined              |no_digits               |email_first|
+------------+------------+---+----------+------+--------------------+------------------------+-----------+
|David Smith |DAVID SMITH |11 |David     |Dav   |David Smith | US    |david.smith#@outlook.com|david      |
|Ravi Garcia |RAVI GARCIA |11 |Ravi      |Rav   |Ravi Garcia         |ravi.garcia#@company.io |ravi       |
|Grace Martin|GRACE MARTIN|12 |Grace     |Gra   |Grace Martin | Japan|grace.martin#@gmail.com |grace      |
|Ravi Lopez  |RAVI LOPEZ  |10 |Ravi      |Rav   |Ravi Lopez | Germany|ravi.lopez#@yahoo.com   |ravi       |
|Farid Lopez |FARID LOPEZ |11 |Farid     |Far   |Farid Lopez | US    |farid.lopez#@outlook.com|farid      |
+------------+------------+---+----------+------+--------------------+------------------------+-----------+
only showing top 5 rows


In [7]:
customers.filter(F.col("email").endswith("@company.io")).select("name", "email").show(3)
customers.filter(F.col("name").like("%Kim%")).select("name").show(3)       # SQL LIKE: % = anything
customers.filter(F.col("name").rlike("^(Alice|Bob) ")).select("name").show(3)  # regex

+------------+--------------------+
|        name|               email|
+------------+--------------------+
| Ravi Garcia|ravi.garcia2@comp...|
|David Garcia|david.garcia6@com...|
|   Ravi Khan|ravi.khan18@compa...|
+------------+--------------------+
only showing top 3 rows
+---------+
|     name|
+---------+
| Olga Kim|
|  Bob Kim|
|Grace Kim|
+---------+
only showing top 3 rows
+----------+
|      name|
+----------+
|Bob Martin|
|   Bob Kim|
| Bob Silva|
+----------+
only showing top 3 rows


## Working with nulls
A very handy idiom: count the nulls in every column in one pass.

In [8]:
customers.select([F.count(F.when(F.col(c).isNull(), 1)).alias(c) for c in customers.columns]).show()

+-----------+----+-----+-------+-----------+
|customer_id|name|email|country|signup_date|
+-----------+----+-----+-------+-----------+
|          0|   0|    0|     15|          0|
+-----------+----+-----+-------+-----------+



In [9]:
customers.fillna({"country": "Unknown"}).filter(F.col("country") == "Unknown").show(3)   # replace nulls
print(customers.dropna(subset=["country"]).count(), "rows with a country")              # drop null rows
customers.select("name", F.coalesce("country", F.lit("??")).alias("country")).show(3)  # first non-null

+-----------+-----------+--------------------+-------+-----------+
|customer_id|       name|               email|country|signup_date|
+-----------+-----------+--------------------+-------+-----------+
|          2|Ravi Garcia|ravi.garcia2@comp...|Unknown| 2023-04-06|
|          9|Sofia Brown|sofia.brown9@gmai...|Unknown| 2023-08-22|
|         44|  Bob Silva|bob.silva44@compa...|Unknown| 2024-02-04|
+-----------+-----------+--------------------+-------+-----------+
only showing top 3 rows
185 rows with a country
+------------+-------+
|        name|country|
+------------+-------+
| David Smith|     US|
| Ravi Garcia|     ??|
|Grace Martin|  Japan|
+------------+-------+
only showing top 3 rows


## Dates and timestamps
`order_ts` was inferred as a `timestamp`. Common functions:

In [10]:
orders.select(
    "order_ts",
    F.to_date("order_ts").alias("date"),
    F.year("order_ts").alias("year"),
    F.month("order_ts").alias("month"),
    F.hour("order_ts").alias("hour"),
    F.dayofweek("order_ts").alias("dow"),                 # 1 = Sunday … 7 = Saturday
    F.date_format("order_ts", "EEEE").alias("day_name"),
    F.date_trunc("month", "order_ts").alias("month_start"),
    F.datediff(F.lit("2025-01-01"), "order_ts").alias("days_to_2025"),
    F.date_add(F.to_date("order_ts"), 30).alias("plus_30d"),
).show(3, truncate=False)

+-------------------+----------+----+-----+----+---+---------+-------------------+------------+----------+
|order_ts           |date      |year|month|hour|dow|day_name |month_start        |days_to_2025|plus_30d  |
+-------------------+----------+----+-----+----+---+---------+-------------------+------------+----------+
|2024-09-11 08:18:52|2024-09-11|2024|9    |8   |4  |Wednesday|2024-09-01 00:00:00|112         |2024-10-11|
|2024-04-18 15:22:01|2024-04-18|2024|4    |15  |5  |Thursday |2024-04-01 00:00:00|258         |2024-05-18|
|2024-04-01 16:43:45|2024-04-01|2024|4    |16  |2  |Monday   |2024-04-01 00:00:00|275         |2024-05-01|
+-------------------+----------+----+-----+----+---+---------+-------------------+------------+----------+
only showing top 3 rows


Parsing strings in non-standard formats — pass a [datetime pattern](https://spark.apache.org/docs/latest/sql-ref-datetime-pattern.html):

In [11]:
spark.createDataFrame([("25/12/2024",), ("01/02/2025",)], "d string") \
     .select("d", F.to_date("d", "dd/MM/yyyy").alias("parsed")).show()

+----------+----------+
|         d|    parsed|
+----------+----------+
|25/12/2024|2024-12-25|
|01/02/2025|2025-02-01|
+----------+----------+



## Applying a function to many columns
`select` accepts a Python list, so list comprehensions are a superpower:

In [12]:
customers.select([F.upper(c).alias(c) if t == "string" else F.col(c) for c, t in customers.dtypes]).show(3)

+-----------+------------+--------------------+-------+-----------+
|customer_id|        name|               email|country|signup_date|
+-----------+------------+--------------------+-------+-----------+
|          1| DAVID SMITH|DAVID.SMITH1@OUTL...|     US| 2023-04-15|
|          2| RAVI GARCIA|RAVI.GARCIA2@COMP...|   NULL| 2023-04-06|
|          3|GRACE MARTIN|GRACE.MARTIN3@GMA...|  JAPAN| 2024-10-27|
+-----------+------------+--------------------+-------+-----------+
only showing top 3 rows


### Exercise 1
Create `with_domain`: `customers` plus a column **`domain`** containing the part of the email **after the `@`** (e.g. `gmail.com`).

*Hint:* `F.split(...).getItem(1)` or `F.regexp_extract` or `F.substring_index`.

In [13]:
with_domain = customers.withColumn("domain", F.split("email", "@").getItem(1))

In [14]:
# Check — run me after your answer
assert "domain" in with_domain.columns
assert with_domain.filter(F.col("domain") == "gmail.com").count() == 48
print("Correct!")

Correct!


### Exercise 2
Create `with_year`: `customers` plus an integer column **`signup_year`**.

In [15]:
with_year = customers.withColumn("signup_year", F.year("signup_date"))

In [16]:
# Check — run me after your answer
assert dict(with_year.dtypes)["signup_year"] == "int"
assert with_year.filter(F.col("signup_year") == 2024).count() == 107
print("Correct!")

Correct!


### Exercise 3
Create `clean_country`: `customers` where null countries are replaced by **`"Unknown"`**.

In [17]:
clean_country = customers.fillna({"country": "Unknown"})

In [18]:
# Check — run me after your answer
assert clean_country.filter(F.col("country").isNull()).count() == 0
assert clean_country.filter(F.col("country") == "Unknown").count() == 15
print("Correct!")

Correct!


### Exercise 4
Create `sized`: `orders` plus a column **`size`**: `"small"` when quantity is 1, `"medium"` for 2–3, `"large"` for 4 or more.

In [19]:
sized = orders.withColumn(
    "size",
    F.when(F.col("quantity") == 1, "small")
     .when(F.col("quantity") <= 3, "medium")
     .otherwise("large"))

In [20]:
# Check — run me after your answer
got = {r["size"]: r["count"] for r in sized.groupBy("size").count().collect()}
assert got == {'small': 2034, 'medium': 1990, 'large': 976}, got
print("Correct!")

Correct!


### Exercise 5
Store in `weekend_orders` the number of orders placed on a **Saturday or Sunday**.

In [21]:
weekend_orders = orders.filter(F.dayofweek("order_ts").isin(1, 7)).count()

In [22]:
# Check — run me after your answer
assert weekend_orders == 1371, weekend_orders
print("Correct!")

Correct!
